Generating Synthetic Data

• Write models that can generate datasets

• Use a variety of models and prompts for diverse outputs

• Create a Gradio Ul for your product



Completed week three exercise:
Created a synthetic dataset generator with a gradio which streams back the data with two open source models, LLaMA 3.2 1B and Microsoft Phi 4 mini for example but you can easily swap and test other models from Hugging Face.

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6


In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, TextIteratorStreamer, BitsAndBytesConfig
import torch
import gc
from IPython.display import Markdown, display, update_display
import gradio as gr
from threading import Thread

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
MODELS = {
      "LLAMA":"meta-llama/Llama-3.2-1B-Instruct",
      "PHI":"microsoft/Phi-4-mini-instruct"
}


GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
MIXTRAL = "mistralai/Mixtral-8x7B-Instruct-v0.1"

In [ ]:
user_prompt = """
    You are a synthetic dataset generator. Generate restaurant reviewers for a pizzeria
    cafe. The reviews should be in mark down review the service and the various amenities like valet and many others
    The rating should be returned in a short sentence and a rating out of 5 stars.

    Reply with only the reviews.
"""


In [ ]:
user_prompt_2 = """
  You are a synthetic dataset generator. Generate exactly 3 high-quality
question-answer pairs about: "Machine Learning".

Requirements:
- Question type: conceptual
- Difficulty: all levels of difficulty low, medium and high
- Each Q&A should be distinct and non-repetitive
- Answers should be accurate, informative, and 1-3 sentences

Respond ONLY with a valid JSON array. No markdown, no explanation. Format:
[
  {"q": "Question text here?", "a": "Answer text here.", "difficulty": "medium"},
]
"""

messages = [
    {"role": "user", "content": user_prompt}
  ]

In [ ]:
# Quantization Config - this allows us to load the model into memory and use less memory

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
tokenizers = {}
models = {}

for name, model_id in MODELS.items():
    tokenizers[name] = AutoTokenizer.from_pretrained(model_id)
    tokenizers[name].pad_token = tokenizers[name].eos_token
    models[name] = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        quantization_config=quant_config
    )

print("Models loaded.")

In [ ]:
def call_models_streaming(prompt):
    print("calling models for streaming")
    global models, tokenizers, MODELS

    streamers = {}
    threads = {}
    messages = [
        {"role": "user", "content": prompt}
    ]

    for model_name, model_id in MODELS.items():
        tokenizer = tokenizers[model_name]
        model = models[model_name]

        streamer = TextIteratorStreamer(
            tokenizer,
            skip_prompt=True,
            skip_special_tokens=True
        )
        streamers[model_name] = streamer

        inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

        generation_kwargs = dict(
            inputs=inputs,
            streamer=streamer,
            max_new_tokens=500,
            do_sample=True,
            temperature=0.7
        )

        thread = Thread(target=model.generate, kwargs=generation_kwargs)
        thread.start()
        threads[model_name] = thread

    llama_current_text = ""
    phi_current_text = ""
    llama_finished = False
    phi_finished = False

    while not (llama_finished and phi_finished):
        llama_token = None
        phi_token = None

        if not llama_finished:
            try:
                llama_token = next(streamers['LLAMA'])
            except StopIteration:
                llama_finished = True

        if not phi_finished:
            try:
                phi_token = next(streamers['PHI'])
            except StopIteration:
                phi_finished = True

        if llama_token:
            llama_current_text += llama_token
        if phi_token:
            phi_current_text += phi_token

        yield llama_current_text, phi_current_text

    for thread in threads.values():
        thread.join()

    print("Streaming completed.")

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# Markdown Comparison UI")

    prompt = gr.Textbox(
        label="Enter Text",
        placeholder="Type something..."
    )

    submit = gr.Button("Submit")

    with gr.Row():
        output1 = gr.Markdown()
        output2 = gr.Markdown()

    submit.click(
        fn=call_models_streaming,
        inputs=prompt,
        outputs=[output1, output2]
    )

demo.launch()

In [ ]:
# Housekeeping  and cleanup

del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()